# 9장. 회귀 분석으로 숫자 예측하기

이 노트북은 `book/chapters/ch09_regression_analysis.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 온라인 쇼핑몰 데이터를 사용해 **주문별 총금액(order_total)** 을 예측하는 회귀 모델링 흐름을 이해하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- 회귀 분석 결과는 `reports/` 폴더에 저장합니다.
- 모델 성능은 MAE, RMSE, R²로 확인합니다.
- `order_total`을 입력값으로 사용하지 않도록 주의합니다. 이것은 데이터 누수입니다.


## 1. 회귀 분석이란 무엇인가

회귀 분석은 연속적인 숫자 값을 예측하는 머신러닝 문제입니다. 분류가 `완료/취소`, `구매/미구매`처럼 범주를 예측한다면, 회귀는 금액, 수량, 점수, 온도처럼 숫자로 표현되는 값을 예측합니다.

이번 장에서는 주문 상세 데이터를 주문 단위로 요약한 뒤, 주문별 총금액을 예측합니다.

| 구분 | 설명 | 이번 실습 예시 |
|---|---|---|
| 입력값(feature) | 예측에 사용할 정보 | 상품 수, 총수량, 평균 단가, 결제수단, 고객 나이 |
| 예측 대상(target) | 모델이 예측할 숫자 | 주문별 총금액 `order_total` |
| 학습 데이터(train) | 모델이 패턴을 배우는 데이터 | 전체 데이터의 80% |
| 테스트 데이터(test) | 학습하지 않은 데이터로 성능 확인 | 전체 데이터의 20% |
| 평가 지표(metric) | 예측이 얼마나 맞는지 확인 | MAE, RMSE, R² |


## 2. 회귀 평가 지표 읽기

회귀 모델은 정확도(accuracy)보다 오차 기반 지표를 사용합니다.

| 지표 | 의미 | 해석 |
|---|---|---|
| MAE | 평균 절대 오차 | 예측값이 실제값과 평균적으로 얼마나 차이 나는지 봅니다. |
| RMSE | 평균 제곱근 오차 | 큰 오차에 더 민감하게 반응합니다. |
| R² | 설명력 | 모델이 실제 값의 변동을 얼마나 설명하는지 봅니다. |

예를 들어 MAE가 12,000이라면 예측 주문 금액이 실제 주문 금액과 평균적으로 약 12,000원 정도 차이 난다고 해석할 수 있습니다.


## 3. 패키지와 경로 설정

scikit-learn을 사용해 선형 회귀와 랜덤 포레스트 회귀 모델을 학습합니다. 노트북 실행 위치가 프로젝트 루트인지 `notebooks/` 폴더인지에 따라 경로를 자동으로 맞춥니다.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


## 4. 전처리 데이터 불러오기

회귀 분석은 5장에서 저장한 전처리 데이터를 사용합니다. 필요한 파일이 없다면 먼저 전처리 스크립트를 실행합니다.

```bash
python scripts/preprocess_data.py
```


In [ ]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, list(customers.columns))
print('orders:', orders.shape, list(orders.columns))
print('order_items:', order_items.shape, list(order_items.columns))


## 5. 주문별 예측 데이터 만들기

하나의 주문에는 여러 상품이 포함될 수 있으므로 주문 상세 데이터를 먼저 주문 단위로 요약합니다. 여기서 `order_total`은 예측 대상입니다.


In [ ]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_features = (
    order_items
    .groupby('order_id', as_index=False)
    .agg(
        item_count=('product_id', 'count'),
        total_quantity=('quantity', 'sum'),
        avg_unit_price=('unit_price', 'mean'),
        order_total=('line_total', 'sum'),
    )
)

order_features.head()


`order_total`은 모델이 맞혀야 하는 정답입니다. 따라서 입력값에는 포함하면 안 됩니다. 입력값 후보는 `item_count`, `total_quantity`, `avg_unit_price`와 주문/고객 정보입니다.


## 6. 주문 정보와 고객 정보 연결하기

주문 날짜에서 월과 요일을 만들고, 결제수단, 주문 상태, 고객 성별, 나이, 도시 정보를 모델 데이터에 연결합니다.


In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['order_month'] = orders['order_date'].dt.month
orders['order_dayofweek'] = orders['order_date'].dt.dayofweek

model_data = order_features.merge(
    orders[['order_id', 'customer_id', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']],
    on='order_id',
    how='left',
)

model_data = model_data.merge(
    customers[['customer_id', 'gender', 'age', 'city']],
    on='customer_id',
    how='left',
)

model_data.head()


In [ ]:
print('모델링 데이터 크기:', model_data.shape)
display(model_data.isna().sum())

model_data = model_data.dropna().copy()
print('결측치 제거 후:', model_data.shape)

model_data['order_total'].describe()


## 7. 입력값과 예측 대상 나누기

이제 모델이 사용할 입력값 `X`와 예측 대상 `y`를 나눕니다. `order_total`은 정답이므로 feature 목록에 들어가면 안 됩니다.


In [ ]:
target = 'order_total'

numeric_features = [
    'item_count',
    'total_quantity',
    'avg_unit_price',
    'order_month',
    'order_dayofweek',
    'age',
]

categorical_features = [
    'payment_method',
    'order_status',
    'gender',
    'city',
]

feature_cols = numeric_features + categorical_features

if target in feature_cols:
    raise ValueError('데이터 누수 위험: order_total이 feature 목록에 포함되어 있습니다.')

X = model_data[feature_cols]
y = model_data[target]

print('X:', X.shape)
print('y:', y.shape)
X.head()


## 8. 범주형 변수 인코딩과 train/test split

문자열 범주형 변수는 모델이 직접 이해하기 어렵기 때문에 OneHotEncoder로 숫자 형태로 변환합니다. 학습 데이터와 테스트 데이터도 나누어야 합니다.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', make_one_hot_encoder(), categorical_features),
        ('num', 'passthrough', numeric_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)


## 9. 선형 회귀 모델 만들기

선형 회귀는 입력값과 예측 대상 사이의 관계를 직선적인 관계로 설명하려는 모델입니다. 구조가 단순하고 해석이 쉬워 회귀 분석의 출발점으로 적합합니다.


In [ ]:
linear_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()),
])

linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, y_pred_linear)
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
linear_r2 = r2_score(y_test, y_pred_linear)

print('Linear Regression MAE:', linear_mae)
print('Linear Regression RMSE:', linear_rmse)
print('Linear Regression R2:', linear_r2)


## 10. 랜덤 포레스트 회귀 모델 만들기

랜덤 포레스트는 여러 개의 의사결정나무를 사용해 예측하는 모델입니다. 선형 회귀보다 복잡한 패턴을 잡을 수 있지만, 해석은 더 어려울 수 있습니다.


In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('cat', make_one_hot_encoder(), categorical_features),
            ('num', 'passthrough', numeric_features),
        ]
    )),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42)),
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print('Random Forest MAE:', rf_mae)
print('Random Forest RMSE:', rf_rmse)
print('Random Forest R2:', rf_r2)


## 11. 모델 비교 결과 저장하기

두 모델의 MAE, RMSE, R²를 비교합니다. 평가 지표가 좋아 보인다고 해서 무조건 좋은 모델은 아닙니다. 해석 가능성, 데이터 크기, 문제 목적도 함께 고려해야 합니다.


In [ ]:
model_comparison = pd.DataFrame({
    'model': ['Linear Regression', 'Random Forest'],
    'MAE': [linear_mae, rf_mae],
    'RMSE': [linear_rmse, rf_rmse],
    'R2': [linear_r2, rf_r2],
}).sort_values('MAE')

model_comparison.to_csv(REPORT_DIR / 'ch09_regression_model_comparison.csv', index=False, encoding='utf-8-sig')
model_comparison


## 12. 실제값과 예측값 비교하기

모델 성능을 숫자로만 보지 말고 실제값과 예측값을 함께 확인합니다. 여기서는 MAE가 낮은 모델의 예측값을 기준으로 오차가 큰 주문을 확인합니다.


In [ ]:
best_model_name = model_comparison.iloc[0]['model']
best_predictions = y_pred_rf if best_model_name == 'Random Forest' else y_pred_linear

prediction_result = X_test.copy()
prediction_result['actual_order_total'] = y_test.values
prediction_result['predicted_order_total'] = best_predictions
prediction_result['error'] = prediction_result['actual_order_total'] - prediction_result['predicted_order_total']
prediction_result['abs_error'] = prediction_result['error'].abs()
prediction_result['model'] = best_model_name
prediction_result = prediction_result.sort_values('abs_error', ascending=False)

prediction_result.to_csv(REPORT_DIR / 'ch09_regression_predictions.csv', index=False, encoding='utf-8-sig')
prediction_result.head(10)


In [ ]:
prediction_result['abs_error'].describe()


## 13. 데이터 누수와 LLM 코드 검토 체크리스트

LLM이 만든 회귀 분석 코드는 오류 없이 실행되어도 데이터 누수나 평가 방식 오류가 있을 수 있습니다. 아래 체크리스트로 검토합니다.


In [ ]:
regression_checklist = pd.DataFrame({
    'check_item': [
        '예측 대상이 명확히 분리되었는가?',
        '정답 컬럼을 입력값으로 사용하지 않았는가?',
        '실제 데이터에 없는 컬럼명을 만들지 않았는가?',
        '범주형 컬럼을 적절히 인코딩했는가?',
        'train/test split을 적용했는가?',
        '테스트 데이터 기준으로 평가했는가?',
        'MAE, RMSE, R²를 함께 확인했는가?',
        '성능 결과를 과장해서 해석하지 않았는가?',
    ],
    'status': ['□'] * 8,
})

regression_checklist.to_csv(REPORT_DIR / 'ch09_regression_checklist.csv', index=False, encoding='utf-8-sig')
regression_checklist


## 14. 회귀 분석 요약 보고서 저장

모델링 데이터 개요, 모델 비교 결과, 예측 오차, 체크리스트, 해석 주의사항을 Markdown 보고서로 저장합니다.


In [ ]:
model_data.to_csv(REPORT_DIR / 'ch09_regression_model_data.csv', index=False, encoding='utf-8-sig')

report_text = f'''# Chapter 9 회귀 분석 요약 보고서

## 1. 분석 목적

온라인 쇼핑몰 주문 데이터를 사용해 주문별 총금액(order_total)을 예측하는 회귀 모델을 만들었습니다.

## 2. 모델링 데이터 개요

- 행 수: {model_data.shape[0]}
- 열 수: {model_data.shape[1]}
- 예측 대상: order_total
- 입력값: {', '.join(feature_cols)}

## 3. 모델 비교 결과

```text
{model_comparison.to_string(index=False)}
```

## 4. 예측 오차 상위 10건

```text
{prediction_result.head(10).to_string(index=False)}
```

## 5. 모델링 검토 체크리스트

```text
{regression_checklist.to_string(index=False)}
```

## 6. 해석 시 주의사항

- MAE는 예측값이 실제 주문 금액과 평균적으로 얼마나 차이 나는지 보여줍니다.
- RMSE는 큰 오차에 더 민감합니다.
- R²는 모델이 실제 값의 변동을 얼마나 설명하는지 보여줍니다.
- 성능이 더 좋아 보이는 모델이 항상 실제 운영에 적합한 모델은 아닙니다.
- order_total 같은 정답 컬럼을 입력값으로 사용하면 데이터 누수가 발생합니다.

## 7. 다음 단계

- 입력값에서 avg_unit_price를 제외했을 때 성능이 어떻게 바뀌는지 비교합니다.
- 예측 오차가 큰 주문 10건의 공통점을 살펴봅니다.
- 고객별 총 구매 금액 예측 또는 상품별 총매출 예측 문제로 확장합니다.
- LLM이 작성한 모델링 코드는 데이터 누수와 평가 방식 중심으로 검토합니다.
'''

report_path = REPORT_DIR / 'ch09_regression_report.md'
report_path.write_text(report_text, encoding='utf-8')

print('회귀 분석 보고서 저장 완료:', report_path)


## 15. 소스 모듈로 전체 회귀 분석 실행

위에서 단계별로 실행한 회귀 분석은 `src/regression.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 실행할 수 있습니다.


In [ ]:
from src.regression import run_regression_analysis

regression_result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

regression_result['model_comparison']


## 16. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 9장 회귀 분석 전체가 자동으로 실행됩니다.

```bash
python scripts/run_regression_analysis.py
```


## 17. LLM에게 회귀 분석 코드를 요청하는 프롬프트

LLM에게 모델링 코드를 요청할 때는 데이터 구조, 예측 대상, 입력값 후보, 검토 기준을 함께 제공합니다.

```text
온라인 쇼핑몰 주문 데이터를 사용해 주문별 총금액을 예측하는 회귀 모델을 만들려고 합니다.

데이터 구조:
- order_items: order_id, product_id, quantity, unit_price, line_total
- orders: order_id, customer_id, order_date, payment_method, order_status
- customers: customer_id, gender, age, city

예측 대상:
- order_total: 주문별 line_total 합계

입력값 후보:
- item_count
- total_quantity
- avg_unit_price
- payment_method
- order_status
- order_month
- order_dayofweek
- gender
- age
- city

요청:
1. 주문별 모델링 데이터셋을 만드는 pandas 코드를 작성해 주세요.
2. train/test split을 적용해 주세요.
3. LinearRegression과 RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R2를 계산해 주세요.
5. 데이터 누수가 발생할 수 있는 부분을 설명해 주세요.

주의:
- 실제 데이터에 없는 컬럼명을 만들지 마세요.
- order_total을 입력값으로 사용하지 마세요.
- 범주형 컬럼은 OneHotEncoder를 사용해 주세요.
- 초보자도 이해할 수 있도록 단계별 설명을 포함해 주세요.
```


## 18. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 입력값에서 `avg_unit_price`를 제외했을 때 성능이 어떻게 바뀌는지 비교하세요.
2. RandomForestRegressor의 `n_estimators` 값을 50, 100, 300으로 바꿔 비교하세요.
3. 예측 오차가 큰 주문 10건의 공통점을 찾아보세요.
4. 고객별 총 구매 금액을 예측하는 모델링 데이터셋을 만들어 보세요.
5. LLM에게 회귀 분석 코드를 작성하게 한 뒤 데이터 누수 여부를 검토하세요.


In [ ]:
# 과제 1. avg_unit_price를 제외한 feature 목록으로 다시 모델을 학습해 보세요.


In [ ]:
# 과제 2. RandomForestRegressor의 n_estimators 값을 바꿔 성능을 비교해 보세요.


## 19. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 회귀 분석의 입력값과 예측 대상 구분
- 주문 상세 데이터를 주문 단위 모델링 데이터로 요약
- 주문 정보와 고객 정보를 연결해 feature 생성
- 숫자형/범주형 feature 구분
- OneHotEncoder와 Pipeline 사용
- train/test split 적용
- Linear Regression과 Random Forest 비교
- MAE, RMSE, R² 해석
- 실제값과 예측값 비교
- 데이터 누수 체크리스트 작성
- `src/regression.py`와 `scripts/run_regression_analysis.py`로 재현 가능한 회귀 분석 구성

다음 장에서는 숫자를 예측하는 회귀와 달리, 특정 상태나 범주를 예측하는 분류 분석을 다룹니다.
